# RSNA Knee — OFFLINE quick-submission notebook
Report labels (local mDeBERTa NLI) -> soft labels -> 2.5D ResNet18 multi-view MIL -> `submission.csv`

**Internet: OFF.** Add ONE input dataset that contains:
- `deberta_nli_offline/` (folder saved with `save_pretrained`)
- `resnet18_timm.pth` (timm ResNet18 state_dict)

Quick-run knobs are in the config cell (`N_WEAK`, `EPOCHS`).

In [1]:
import os, re, gc, time, glob, random, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np, pandas as pd, pydicom, cv2, torch, torch.nn as nn, timm
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = "/kaggle/working"
LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
VIEWS = ["Sagittal","Coronal","Axial"]

# ---------------- QUICK-RUN CONFIG ----------------
N_WEAK   = 600     # unlabeled (report-only) studies to auto-label + train on. More = better but slower
EPOCHS   = 4
N_SLICES = 16
IMG_SIZE = 176
CROP     = 160
BATCH    = 8
LR       = 3e-4
SEED     = 42
TRAIN_ON_GOLD = False   # False: 58 gold studies kept as honest test. True: also train on them
NUM_WORKERS = min(4, os.cpu_count() or 1)
# --------------------------------------------------

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
print("Device:", DEVICE, "| CPUs:", os.cpu_count())

Device: cuda | CPUs: 4


## 1. Locate inputs (no internet, no hard-coded dataset names)

In [2]:
SKIP = {"train_series", "test_series"}   # never walk the huge DICOM folders

def find_path(match, max_depth=5, root="/kaggle/input"):
    for cur, dirs, files in os.walk(root):
        depth = cur[len(root):].count(os.sep)
        dirs[:] = [d for d in dirs if d not in SKIP] if depth < max_depth else []
        r = match(cur, files)
        if r: return r
    return None

DATA_DIR = find_path(lambda cur, files: cur if "train.csv" in files and "train_series.csv" in files else None)
assert DATA_DIR, "Competition data not found under /kaggle/input"

def glob_first(name):
    for depth in range(1, 6):                       # /kaggle/input/*/.../<name>
        hits = glob.glob("/kaggle/input/" + "*/" * depth + name)
        if hits:
            return hits[0]
    return None

NLI_DIR  = glob_first("deberta_nli_offline")
RESNET_W = glob_first("resnet18_timm.pth")

if NLI_DIR is None or RESNET_W is None:             # debug listing
    for p in glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*") + glob.glob("/kaggle/input/*/*/*"):
        print(p)
assert NLI_DIR, "deberta_nli_offline not found - see listing above"

print("DATA_DIR  :", DATA_DIR); print("NLI_DIR   :", NLI_DIR); print("RESNET_W  :", RESNET_W)
if RESNET_W is None:
    print("WARNING: resnet weights not found -> random init (much worse accuracy)")

DATA_DIR  : /kaggle/input/competitions/rsna-knee-abnormality-detection
NLI_DIR   : /kaggle/input/datasets/amudhana/models/deberta_nli_offline
RESNET_W  : /kaggle/input/datasets/amudhana/models/resnet18_timm.pth


## 2. Load CSVs

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
train_series = pd.read_csv(f"{DATA_DIR}/train_series.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
test_series = pd.read_csv(f"{DATA_DIR}/test_series.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(train.shape, train_series.shape, test.shape, test_series.shape)

has_labels = train[LABEL_COLS].notna().all(axis=1)
gold_df = train[has_labels]
unl_df = train[~has_labels]
print(f"gold={len(gold_df)}  unlabeled={len(unl_df)}")

# subsample: ALL gold + N_WEAK random unlabeled studies
sub = pd.concat([gold_df, unl_df.sample(n=min(N_WEAK, len(unl_df)), random_state=SEED)]).reset_index(drop=True)
gold_mask = sub[LABEL_COLS].notna().all(axis=1).values
print("subsample:", len(sub), "| gold in it:", gold_mask.sum())

(4407, 14) (24371, 5) (3, 1) (15, 5)
gold=58  unlabeled=4349
subsample: 658 | gold in it: 58


## 3. Weak labelling with the LOCAL NLI model
Report -> sentences -> NLI(sentence, hypothesis) -> max over sentences -> Platt calibration on gold -> soft labels.

In [4]:
def clean_report(t):
    if not isinstance(t, str): return ""
    t = re.sub(r"\[.*?\]", "", t)
    return re.sub(r"\s+", " ", t).strip()

def split_sentences(text, min_chars=3):
    parts = re.split(r"(?<=[.;!?])\s+|\n+", str(text))
    out, seen = [], set()
    for p in parts:
        p = p.strip(" .-"); k = p.lower()
        if len(p) >= min_chars and k not in seen:
            seen.add(k); out.append(p)
    return out if out else [str(text)[:1000] or "empty report"]

HYPOTHESES = {
 "ACL": ["The anterior cruciate ligament is torn.", "There is an injury of the anterior cruciate ligament."],
 "MCL": ["The medial collateral ligament is torn or sprained."],
 "Medial Meniscus": ["There is a tear of the medial meniscus."],
 "Lateral Meniscus": ["There is a tear of the lateral meniscus."],
 "Medial OA": ["There is osteoarthritis of the medial femorotibial compartment."],
 "Lateral OA": ["There is osteoarthritis of the lateral femorotibial compartment."],
 "PF OA": ["There is patellofemoral osteoarthritis or chondromalacia of the patella."],
 "Effusion": ["There is a joint effusion."],
 "Synovitis": ["There is synovitis."],
 "Baker's": ["There is a Baker's cyst."],
 "Contusion": ["There is a bone contusion or bone bruise."],
 "Fracture": ["There is a fracture."],
}
HYP_FLAT = [(lab, h) for lab in LABEL_COLS for h in HYPOTHESES[lab]]

sub["sentences"] = sub["Report"].apply(clean_report).apply(split_sentences)
print("NLI pairs:", int(sub["sentences"].str.len().sum()) * len(HYP_FLAT))

@torch.no_grad()
def nli_scores(sentences, hyps, sent_batch=32, max_len=160):
    tok = AutoTokenizer.from_pretrained(NLI_DIR, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(NLI_DIR, local_files_only=True).to(DEVICE).eval()
    if DEVICE == "cuda": model.half()

    print("NLI labels:", model.config.id2label)
    l2i = {k.lower(): v for k, v in model.config.label2id.items()}
    if "entailment" in l2i and "contradiction" in l2i:
        cols = [l2i["entailment"], l2i["contradiction"]]        # 3-class: entailment vs contradiction
    elif "entailment" in l2i:
        cols = [l2i["entailment"]] + [i for i in range(model.config.num_labels) if i != l2i["entailment"]]  # binary: entail vs not_entail
    else:
        cols = [0, 2]                                           # generic LABEL_0/1/2 -> mDeBERTa order (0=entail, 2=contradict)
    print("Using logit columns [entail, other]:", cols)

    H = len(hyps); out = np.zeros((len(sentences), H), np.float32)
    order = np.argsort([len(s) for s in sentences])
    for st in tqdm(range(0, len(order), sent_batch), desc="NLI"):
        idx = order[st:st + sent_batch]
        prem = [sentences[i] for i in idx for _ in range(H)]
        hyp = list(hyps) * len(idx)
        enc = tok(prem, hyp, truncation="only_first", max_length=max_len, padding=True, return_tensors="pt").to(DEVICE)
        lg = model(**enc).logits.float()
        out[idx] = torch.softmax(lg[:, cols], -1)[:, 0].view(len(idx), H).cpu().numpy()
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return out

flat, owner = [], []
for si, ss in enumerate(sub["sentences"]):
    flat.extend(ss); owner.extend([si] * len(ss))
owner = np.array(owner)

t0 = time.time()
sent_sc = nli_scores(flat, [h for _, h in HYP_FLAT])
print(f"NLI done in {(time.time()-t0)/60:.1f} min")

hyp_sc = np.zeros((len(sub), len(HYP_FLAT)), np.float32)
np.maximum.at(hyp_sc, owner, sent_sc)
study_scores = np.stack([hyp_sc[:, [j for j, (l, _) in enumerate(HYP_FLAT) if l == lab]].max(1) for lab in LABEL_COLS], 1)

NLI pairs: 131365


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI labels: {0: 'entailment', 1: 'not_entailment'}
Using logit columns [entail, other]: [0, 1]


NLI:   0%|          | 0/316 [00:00<?, ?it/s]

NLI done in 1.3 min


In [5]:
def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan

Xg = study_scores[gold_mask]
Yg = sub.loc[gold_mask, LABEL_COLS].values.astype(int)
print("Report-labeller AUC on gold (raw NLI score):")
lab_auc = {lab: safe_auc(Yg[:, j], Xg[:, j]) for j, lab in enumerate(LABEL_COLS)}
print(pd.Series(lab_auc).round(3).to_string()); print("macro:", round(np.nanmean(list(lab_auc.values())), 4))

def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4); return np.log(p / (1 - p))

# Platt calibration per label (falls back to raw score if gold has <2 positives/negatives)
soft = np.zeros_like(study_scores)
for j, lab in enumerate(LABEL_COLS):
    y = Yg[:, j]
    if y.sum() >= 2 and (1 - y).sum() >= 2:
        cal = LogisticRegression().fit(logit(Xg[:, [j]]), y)
        soft[:, j] = cal.predict_proba(logit(study_scores[:, [j]]))[:, 1]
    else:
        soft[:, j] = study_scores[:, j]
soft[gold_mask] = Yg           # gold studies keep true labels

labels_df = pd.DataFrame(soft, columns=LABEL_COLS)
labels_df.insert(0, "StudyInstanceUID", sub["StudyInstanceUID"].values)
labels_df["is_gold"] = gold_mask
print(pd.DataFrame({"gold": Yg.mean(0), "weak": soft[~gold_mask].mean(0)}, index=LABEL_COLS).round(3))

Report-labeller AUC on gold (raw NLI score):
ACL                 0.541
MCL                 0.537
Medial Meniscus     0.739
Lateral Meniscus    0.697
Medial OA           0.641
Lateral OA          0.836
PF OA               0.563
Effusion            0.579
Synovitis           0.651
Baker's             0.733
Contusion           0.698
Fracture            0.683
macro: 0.6581
                   gold   weak
ACL               0.414  0.400
MCL               0.155  0.140
Medial Meniscus   0.448  0.383
Lateral Meniscus  0.397  0.307
Medial OA         0.259  0.237
Lateral OA        0.190  0.196
PF OA             0.362  0.353
Effusion          0.603  0.575
Synovitis         0.466  0.420
Baker's           0.207  0.214
Contusion         0.328  0.272
Fracture          0.310  0.231


## 4. MRI loading (decoded once into RAM for the subsample — no disk cache)

In [6]:
def pick_view_series(rows, plane):
    r = rows[rows["Anatomical_Plane"] == plane]
    if len(r) == 0: return None
    f = r[r["Fluid_Sensitive"] == 1]
    return (f if len(f) else r).iloc[0]["SeriesInstanceUID"]

def sorted_files(d):
    files = [f for f in os.listdir(d) if not f.startswith(".")]
    stems = [os.path.splitext(f)[0] for f in files]
    if all(s.isdigit() for s in stems):
        return [f for _, f in sorted(zip(map(int, stems), files))]
    keyed = []
    for f in files:
        try: n = int(pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True).InstanceNumber)
        except Exception: n = 0
        keyed.append((n, f))
    return [f for _, f in sorted(keyed)]

def load_volume_u8(split, study, series):
    d = f"{DATA_DIR}/{split}_series/{study}/{series}"
    files = sorted_files(d)
    idx = np.linspace(0, len(files) - 1, N_SLICES).round().astype(int)
    imgs = []
    for i in idx:
        try:
            a = pydicom.dcmread(os.path.join(d, files[i])).pixel_array.astype(np.float32)
            if a.ndim == 3: a = a[a.shape[0] // 2]
            a = cv2.resize(a, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        except Exception:
            a = np.zeros((IMG_SIZE, IMG_SIZE), np.float32)
        imgs.append(a)
    v = np.stack(imgs)
    lo, hi = np.percentile(v, [1, 99.5])
    return (np.clip((v - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)

def build_study(split, study, rows):
    vol = np.zeros((len(VIEWS), N_SLICES, IMG_SIZE, IMG_SIZE), np.uint8)
    mask = np.zeros(len(VIEWS), np.float32)
    for v, plane in enumerate(VIEWS):
        s = pick_view_series(rows, plane)
        if s is None: continue
        try: vol[v] = load_volume_u8(split, study, s); mask[v] = 1.0
        except Exception: pass
    if mask.sum() == 0: mask[0] = 1.0
    return vol, mask

ser_train = {k: g for k, g in train_series.groupby("StudyInstanceUID")}
ser_test = {k: g for k, g in test_series.groupby("StudyInstanceUID")}
EMPTY = train_series.iloc[0:0]

t0 = time.time()
res = Parallel(n_jobs=os.cpu_count(), backend="loky")(
    delayed(build_study)("train", u, ser_train.get(u, EMPTY)) for u in tqdm(sub["StudyInstanceUID"], desc="decode train"))
VOLS = np.stack([r[0] for r in res]); MASKS = np.stack([r[1] for r in res]); del res
print(f"decoded {VOLS.shape} in {(time.time()-t0)/60:.1f} min | RAM {VOLS.nbytes/1e9:.2f} GB")
print("all-zero volumes:", int((VOLS.reshape(len(VOLS), -1).max(1) == 0).sum()), "(should be ~0)")

decode train:   0%|          | 0/658 [00:00<?, ?it/s]

decoded (658, 3, 16, 176, 176) in 5.2 min | RAM 0.98 GB
all-zero volumes: 0 (should be ~0)


## 5. Model: 2.5D ResNet18 + per-label attention over slices/views (offline weights)

In [7]:
def crop(vol, c, rand):
    H, W = vol.shape[-2:]
    if rand: y0, x0 = np.random.randint(0, H - c + 1), np.random.randint(0, W - c + 1)
    else:    y0, x0 = (H - c) // 2, (W - c) // 2
    return vol[..., y0:y0 + c, x0:x0 + c]

class TrainDS(torch.utils.data.Dataset):
    def __init__(self, idx, targets, train): self.idx, self.t, self.train = list(idx), targets.astype(np.float32), train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        v = torch.from_numpy(VOLS[self.idx[i]]).float() / 255.0
        v = crop(v, CROP, self.train)
        if self.train: v = (v * np.random.uniform(0.9, 1.1) + np.random.uniform(-0.05, 0.05)).clamp(0, 1)
        return v, torch.from_numpy(MASKS[self.idx[i]]), torch.from_numpy(self.t[i])

class TestDS(torch.utils.data.Dataset):      # decodes DICOMs on the fly inside DataLoader workers
    def __init__(self, uids): self.uids = list(uids)
    def __len__(self): return len(self.uids)
    def __getitem__(self, i):
        u = self.uids[i]
        vol, mask = build_study("test", u, ser_test.get(u, EMPTY))
        return crop(torch.from_numpy(vol).float() / 255.0, CROP, False), torch.from_numpy(mask)

class MultiViewMIL(nn.Module):
    def __init__(self, weights=None, n_views=3, n_labels=len(LABEL_COLS), drop=0.3):
        super().__init__()
        self.encoder = timm.create_model("resnet18", pretrained=False, num_classes=0, in_chans=3)
        if weights:
            sd = torch.load(weights, map_location="cpu")
            sd = sd.get("state_dict", sd)
            print("load resnet:", self.encoder.load_state_dict(sd, strict=False))
        D = self.encoder.num_features
        self.view_emb = nn.Parameter(torch.zeros(n_views, D))
        self.att = nn.Sequential(nn.Linear(D, 256), nn.Tanh(), nn.Linear(256, n_labels))
        self.drop = nn.Dropout(drop)
        self.cls_w = nn.Parameter(torch.randn(n_labels, D) * 0.01)
        self.cls_b = nn.Parameter(torch.zeros(n_labels))

    @staticmethod
    def to_25d(x):
        prev = torch.cat([x[:, :, :1], x[:, :, :-1]], 2); nxt = torch.cat([x[:, :, 1:], x[:, :, -1:]], 2)
        return torch.stack([prev, x, nxt], 3)

    def forward(self, x, view_mask):
        B, V, S, H, W = x.shape
        x = (self.to_25d(x) - 0.45) / 0.225
        f = self.encoder(x.reshape(B * V * S, 3, H, W)).reshape(B, V, S, -1) + self.view_emb[None, :, None, :]
        tok = f.reshape(B, V * S, -1)
        tm = view_mask[:, :, None].expand(B, V, S).reshape(B, V * S)
        a = self.att(tok).float().masked_fill(tm[..., None] == 0, -1e4)
        a = torch.softmax(a, 1)
        pooled = torch.einsum("btl,btd->bld", a, self.drop(tok).float())
        return (pooled * self.cls_w).sum(-1) + self.cls_b

def macro_auc(y, p):
    a = [safe_auc(y[:, j], p[:, j]) for j in range(y.shape[1])]
    return np.nanmean(a), a

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for b in loader:
        x, m = b[0], b[1]
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            lg = model(x.to(DEVICE), m.to(DEVICE))
        out.append(torch.sigmoid(lg.float()).cpu().numpy())
    return np.concatenate(out)

## 6. Train (gold = held-out test unless TRAIN_ON_GOLD)

In [8]:
Y = labels_df[LABEL_COLS].values.astype(np.float32)
gold_idx = np.where(gold_mask)[0]
weak_idx = np.where(~gold_mask)[0]
rng = np.random.RandomState(SEED); rng.shuffle(weak_idx)
n_val = max(20, int(0.1 * len(weak_idx)))
val_idx, tr_idx = weak_idx[:n_val], weak_idx[n_val:]
if TRAIN_ON_GOLD: tr_idx = np.concatenate([tr_idx, gold_idx])
print(f"train={len(tr_idx)} val(weak)={len(val_idx)} gold={len(gold_idx)}")

def loader(idx, train):
    return torch.utils.data.DataLoader(TrainDS(idx, Y[idx], train), batch_size=BATCH, shuffle=train,
                                       num_workers=NUM_WORKERS, pin_memory=True, drop_last=train)
tr_ld, val_ld, gold_ld = loader(tr_idx, True), loader(val_idx, False), loader(gold_idx, False)

model = MultiViewMIL(RESNET_W).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
total = EPOCHS * len(tr_ld)
sched = get_cosine_schedule_with_warmup(opt, int(0.05 * total), total)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
crit = nn.BCEWithLogitsLoss()
CKPT = f"{WORK_DIR}/best_model.pth"
best = -1

for ep in range(EPOCHS):
    model.train(); run = 0
    for x, m, y in tqdm(tr_ld, desc=f"epoch {ep+1}/{EPOCHS}"):
        x, m, y = x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            lg = model(x, m)
        loss = crit(lg.float(), y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
    v_auc, _ = macro_auc((Y[val_idx] >= 0.5).astype(int), predict(model, val_ld))
    g_auc, _ = macro_auc(Y[gold_idx].astype(int), predict(model, gold_ld))   # monitoring only
    print(f"ep{ep+1} loss={run/len(tr_ld):.4f} val_auc={v_auc:.4f} gold_auc={g_auc:.4f}")
    if np.nan_to_num(v_auc) >= best:
        best = np.nan_to_num(v_auc); torch.save(model.state_dict(), CKPT)

model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
del VOLS; gc.collect()

train=540 val(weak)=60 gold=58
load resnet: _IncompatibleKeys(missing_keys=[], unexpected_keys=['fc.weight', 'fc.bias'])


epoch 1/4:   0%|          | 0/67 [00:00<?, ?it/s]

ep1 loss=0.6103 val_auc=0.5939 gold_auc=0.5366


epoch 2/4:   0%|          | 0/67 [00:00<?, ?it/s]

ep2 loss=0.5842 val_auc=0.6288 gold_auc=0.5657


epoch 3/4:   0%|          | 0/67 [00:00<?, ?it/s]

ep3 loss=0.5822 val_auc=0.6444 gold_auc=0.5631


epoch 4/4:   0%|          | 0/67 [00:00<?, ?it/s]

ep4 loss=0.5811 val_auc=0.6531 gold_auc=0.5639


477

## 7. Inference on test -> `/kaggle/working/submission.csv`

In [9]:
test_ld = torch.utils.data.DataLoader(TestDS(test["StudyInstanceUID"]), batch_size=BATCH, shuffle=False,
                                      num_workers=NUM_WORKERS, pin_memory=True)
t0 = time.time()
pred = predict(model, test_ld)
print(f"test inference {(time.time()-t0)/60:.1f} min")

submission = pd.DataFrame(pred, columns=LABEL_COLS)
submission.insert(0, "StudyInstanceUID", test["StudyInstanceUID"].values)
submission = submission[sample_submission.columns]
submission[LABEL_COLS] = submission[LABEL_COLS].fillna(0.5).clip(0, 1)
assert len(submission) == len(test) and list(submission.columns) == list(sample_submission.columns)
submission.to_csv(f"{WORK_DIR}/submission.csv", index=False)
print(submission.shape); submission.head()

test inference 0.2 min
(3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.414829,0.223767,0.411922,0.344731,0.298190,0.286237,0.382396,0.545904,0.440482,0.261077,0.32457,0.299342
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.424242,0.203824,0.450784,0.380588,0.300001,0.291741,0.391970,0.573408,0.450878,0.276113,0.32666,0.288109
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.407759,0.159853,0.395207,0.314337,0.250534,0.218948,0.358505,0.549619,0.423066,0.210942,0.28524,0.251475
